In [ ]:
# =========================
# 1. Install Dependencies
# =========================
!pip -q install google-generativeai faiss-cpu sentence-transformers PyPDF2

# =========================
# 2. Import Libraries
# =========================
import os
import faiss
import numpy as np
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
from google.colab import drive
from google.colab import userdata

# =========================
# 3. Mount Google Drive
# =========================
drive.mount('/content/drive')

# =========================
# 4. Provide PDF Path
# =========================
# Example: /content/drive/MyDrive/sample.pdf
pdf_path = input("Enter full PDF path from Google Drive: ")

if not os.path.exists(pdf_path):
    raise FileNotFoundError("Invalid path! Please check your file path.")

# =========================
# 5. Configure Gemini API Key
# =========================
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

# =========================
# 6. Load PDF
# =========================
reader = PdfReader(pdf_path)
text = ""

for page in reader.pages:
    text += page.extract_text() or ""

print("PDF loaded successfully!")

# =========================
# 7. Split Text into Chunks
# =========================
def split_text(text, chunk_size=300, overlap=50):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks

chunks = split_text(text)
print(f"Total chunks: {len(chunks)}")

# =========================
# 8. Generate Embeddings
# =========================
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks)

# =========================
# 9. Store in FAISS
# =========================
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# =========================
# 10. Initialize Gemini Model
# =========================
model = genai.GenerativeModel("gemini-2.5-flash-lite")

# =========================
# 11. Retrieval Function
# =========================
def retrieve(query, k=3):
    query_vec = embedder.encode([query])
    distances, indices = index.search(np.array(query_vec), k)
    return [chunks[i] for i in indices[0]]

# =========================
# 12. RAG Pipeline
# =========================
def rag_pipeline(query):
    retrieved_chunks = retrieve(query)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
    Answer the question ONLY using the context below.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = model.generate_content(prompt)
    return context, response.text

# =========================
# 13. User Query
# =========================
query = input("Enter your question: ")

# =========================
# 14. Generate Response
# =========================
context, answer = rag_pipeline(query)

# =========================
# 15. Display Output
# =========================
print("\n===== Retrieved Context =====\n")
print(context)

print("\n===== Generated Answer =====\n")
print(answer)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter full PDF path from Google Drive: /content/drive/MyDrive/NIPS-2017-attention-is-all-you-need-Paper.pdf
PDF loaded successfully!
Total chunks: 131


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Enter your question: What is a Transfromer?

===== Retrieved Context =====

 shown to perform well on simple-language question answering and
language modeling tasks [28].
To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned R

Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to

input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [14, 15] and [8].
3 Model Architecture
Most competitive neural sequence transduction models have a

===== Generated An